# Stored CoT answer evaluation
Run after Notebook 03 using the updated repository. Incomplete generations are excluded from answer evaluations and accuracy; their IDs are saved for auditing. No GPU is needed.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
from google.colab import drive

drive.mount("/content/drive")
os.chdir("/content/PSSN-LLM-Reasoning")
from src.evaluation import evaluate_records, summarize_accuracy
from src.utils import ExperimentLogger

root = Path("/content/drive/MyDrive/PSSN2/artifacts/cot_baseline")
candidates = sorted(root.glob("*/cot_baseline_results.jsonl"))
assert candidates, "Run Notebook 03 first."
source = candidates[-1]
records = [json.loads(line) for line in source.read_text(encoding="utf-8").splitlines() if line.strip()]
evaluations = evaluate_records(records)
summary = summarize_accuracy(evaluations)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = source.parent / "evaluations" / stamp
logger = ExperimentLogger(
    output_dir / "answer_evaluations.jsonl",
    experiment_name="gsm8k-answer-evaluation",
    run_metadata={"source_results": str(source), "source_run_ids": sorted({r["run_id"] for r in records})},
)
logger.output_path.parent.mkdir(parents=True, exist_ok=True)
logger.output_path.touch()
logger.log_many(result for result in evaluations if result["is_correct"] is not None)
excluded = [result["problem_id"] for result in evaluations if result["is_correct"] is None]
(output_dir / "excluded_problem_ids.json").write_text(json.dumps(excluded, indent=2) + "\n", encoding="utf-8")
summary.update(evaluation_run_id=logger.run_id, source_results=str(source))
(output_dir / "accuracy_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
configuration = {
    "token_limit_policy": "Exclude length stops from scored accuracy",
    "legacy_policy": "Exclude records at their cap when EOS is not recorded",
    "accuracy_denominator": "scored_problems",
    "source_results": str(source),
}
(output_dir / "evaluation_configuration.json").write_text(json.dumps(configuration, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))
for result in evaluations:
    if result["is_correct"] is None:
        continue
    print(result["problem_id"], result["evaluation_status"], result["predicted_answer"], result["is_correct"])
print("Saved evaluation:", output_dir)